## <small>
Copyright (c) 2017-21 Andrew Glassner

Permission is hereby granted, free of charge, to any person obtaining a copy of this software and associated documentation files (the "Software"), to deal in the Software without restriction, including without limitation the rights to use, copy, modify, merge, publish, distribute, sublicense, and/or sell copies of the Software, and to permit persons to whom the Software is furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.
</small>



# Deep Learning: A Visual Approach
## by Andrew Glassner, https://glassner.com
### Order: https://nostarch.com/deep-learning-visual-approach
### GitHub: https://github.com/blueberrymusic
------

### What's in this notebook

This notebook is provided as a “behind-the-scenes” look at code used to make some of the figures in this chapter. It is cleaned up a bit from the original code that I hacked together, and is only lightly commented. I wrote the code to be easy to interpret and understand, even for those who are new to Python. I tried never to be clever or even more efficient at the cost of being harder to understand. The code is in Python3, using the versions of libraries as of April 2021.

This notebook may contain additional code to create models and images not in the book. That material is included here to demonstrate additional techniques.

Note that I've included the output cells in this saved notebook, but Jupyter doesn't save the variables or data that were used to generate them. To recreate any cell's output, evaluate all the cells from the start up to that cell. A convenient way to experiment is to first choose "Restart & Run All" from the Kernel menu, so that everything's been defined and is up to date. Then you can experiment using the variables, data, functions, and other stuff defined in this notebook.

## Chapter 19: RNNs - Notebook 3: Holmes

This notebook is a bit more casual than most of the notebooks in this repo,
as it's only meant to do a single specific thing. There's no organization
into useful functions and subroutines - it's just one cell after another,
computing things in sequence! Feel free to re-organize it if you'd like to
do more general-purpose text generation.

In [1]:
from keras.models import Sequential
from keras.layers import Dense, Activation
from keras.layers import LSTM
from keras.optimizers import RMSprop
import numpy as np
import random
import sys

In [2]:
# Workaround for Keras issues on Mac computers (you can comment this
# out if you're not on a Mac, or not having problems)
import os
os.environ['KMP_DUPLICATE_LIB_OK']='True'

In [8]:
import os
import requests

# Ensure folder exists
os.makedirs("input_data", exist_ok=True)

# Project Gutenberg Sherlock Holmes text (The Adventures of Sherlock Holmes)
holmes_url = "https://www.gutenberg.org/cache/epub/1661/pg1661.txt"

holmes_path = "input_data/holmes.txt"
test_holmes_path = "input_data/test-holmes.txt"

def download_if_needed(url, path):
    if os.path.exists(path):
        print(f"Already exists: {path}")
        return
    print(f"Downloading {url} -> {path}")
    resp = requests.get(url)
    resp.raise_for_status()
    with open(path, "w", encoding="utf8") as f:
        f.write(resp.text)
    print("Saved:", path)

download_if_needed(holmes_url, holmes_path)
# For simplicity, reuse same text as test file
if not os.path.exists(test_holmes_path):
    print(f"Copying {holmes_path} -> {test_holmes_path}")
    with open(holmes_path, "r", encoding="utf8") as fin, \
         open(test_holmes_path, "w", encoding="utf8") as fout:
        fout.write(fin.read())

print("Files ready:",
      os.path.exists(holmes_path),
      os.path.exists(test_holmes_path))

Saved: input_data/holmes.txt
Copying input_data/holmes.txt -> input_data/test-holmes.txt
Files ready: True True


In [10]:
import os
import sys
import random
import numpy as np

import matplotlib.pyplot as plt  # optional, but often used
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import RMSprop

# ------------------------------------------------------------------
# Download files if needed
# ------------------------------------------------------------------
if not os.path.exists("input_data"):
    os.makedirs("input_data")

# Download the input files if they don't exist
!wget -nc https://raw.githubusercontent.com/blueberrymusic/deep-learning-visual-approach/main/Chap19/input_data/holmes.txt -P input_data/
!wget -nc https://raw.githubusercontent.com/blueberrymusic/deep-learning-visual-approach/main/Chap19/input_data/test-holmes.txt -P input_data/

window_length = 40
window_step = 3
number_of_epochs = 2
generated_text_length = 1000
batch_size = 100
test_input_file = "input_data/test-holmes.txt"
input_file = "input_data/holmes.txt"
output_file = "holmes-by-char.txt"

# ------------------------------------------------------------------
# Read and preprocess text
# ------------------------------------------------------------------
with open(input_file, "r", encoding="utf8") as f:
    text = f.read()

# replace newlines with blanks, and double blanks with singles
text = text.replace("\n", " ")
text = text.replace("  ", " ")
print("corpus length:", len(text))

chars = sorted(list(set(text)))
print("total chars:", len(chars))
char_indices = {c: i for i, c in enumerate(chars)}
indices_char = {i: c for i, c in enumerate(chars)}

# ------------------------------------------------------------------
# Build training sequences
# ------------------------------------------------------------------
sentences = []
next_chars = []
for i in range(0, len(text) - window_length, window_step):
    sentences.append(text[i: i + window_length])
    next_chars.append(text[i + window_length])
print("number of sequences of", window_length, ":", len(sentences))

# Turn inputs and targets into one-hot versions
X = np.zeros((len(sentences), window_length, len(chars)), dtype=bool)
y = np.zeros((len(sentences), len(chars)), dtype=bool)
for i, sentence in enumerate(sentences):
    for t, char in enumerate(sentence):
        X[i, t, char_indices[char]] = True
    y[i, char_indices[next_chars[i]]] = True

# ------------------------------------------------------------------
# Build the model
# ------------------------------------------------------------------
print("Build model...")
model = Sequential()
model.add(LSTM(128, return_sequences=True,
               input_shape=(window_length, len(chars))))
model.add(LSTM(128))
model.add(Dense(len(chars), activation="softmax"))

# lr argument is 'learning_rate' in modern Keras
optimizer = RMSprop(learning_rate=0.01)
model.compile(loss="categorical_crossentropy", optimizer=optimizer)

model.summary()

# ------------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------------
def sample(preds, temperature=1.0):
    """Sample an index from a probability array."""
    preds = np.asarray(preds).astype("float64")
    preds = np.log(preds + 1e-8) / temperature  # avoid log(0)
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    probas = np.random.multinomial(1, preds, 1)
    return np.argmax(probas)

def print_string(out_str=""):
    print(out_str, end="")
    File_writer.write(out_str)

# ------------------------------------------------------------------
# Training loop with text generation
# ------------------------------------------------------------------
File_writer = open(output_file, "w", encoding="utf8")

for iteration in range(number_of_epochs):
    print_string("--------------------------------------------------\n")
    print_string(f"Iteration {iteration}\n")

    history = model.fit(
        X, y,
        batch_size=batch_size,
        epochs=1,
        verbose=1
    )
    loss_val = history.history["loss"][-1]
    print_string(f"  Loss from iteration {iteration} = {loss_val}\n")

    start_index = random.randint(0, len(text) - window_length - 1)

    for diversity in [0.5, 1.0, 1.5, 2.0]:
        print_string(f"\n----- diversity: {diversity}\n")

        generated = ""
        sentence = text[start_index: start_index + window_length]
        generated += sentence
        print_string(f"----- Generating with seed: <{sentence}>\n")

        for _ in range(generated_text_length):
            x = np.zeros((1, window_length, len(chars)))
            for t, char in enumerate(sentence):
                x[0, t, char_indices[char]] = 1.0

            preds = model.predict(x, verbose=0)[0]
            next_index = sample(preds, diversity)
            next_char = indices_char[next_index]

            generated += next_char
            sentence = sentence[1:] + next_char

        print_string(generated + "\n\n")
        File_writer.flush()

print_string("\n")
File_writer.close()

File ‘input_data/holmes.txt’ already there; not retrieving.

File ‘input_data/test-holmes.txt’ already there; not retrieving.

corpus length: 578647
total chars: 98
number of sequences of 40 : 192869
Build model...


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 40, 128)        │       116,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 98)             │        12,642 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 260,450 (1017.38 KB)

 Trainable params: 260,450 (1017.38 KB)

 Non-trainable params: 0 (0.00 B)

--------------------------------------------------
Iteration 0
1929/1929 ━━━━━━━━━━━━━━━━━━━━ 416s 214ms/step - loss: 2.1384
  Loss from iteration 0 = 2.138362169265747

----- diversity: 0.5
----- Generating with seed: <der his ear, while all the tags of his h>
der his ear, while all the tags of his his way a starce of you that I had not a deand that it is the that the everestan and cate, and the had be you have she had every that why showever been from his read at the dation of the inder see the still of for you will that the dest it is of the look of good that a strode have one that the seem and ever of a stinger that the dats, and I was the heads, starks, and he said and the shard of the lood.” “When her was of the wate that the tnook in the cone that he was the lattle lade good of the his will than the latt the hair the matter, and the evershad of his look that I fat his like add the stry course to the shour to the tage stards the ster of the stare the ster straech apon the for and